In [ ]:
import numpy as np
from tqdm.notebook import tqdm
from numpy.random import default_rng

So we split up the problem in two subproblems.

Problem 1: we need to find 3 sets of 20 pairs that roughly correspond to the given joint probability matrix. Moreover, over all 60 pairs, their frequencies should _exactly_ correspond to the joint probability matrix

Problem 2: we need to make sure that there are any repetitions in either of the dimensions

# Problem 1

We need 3 sub co-occurence matrices that add up to the following summed matrix

In [ ]:
jp = np.array([[0.5, 0.35, 0.15],
               [0.15, 0.5, 0.35],
               [0.35, 0.15, 0.5]])
jp = jp/3.
n = jp*60
n = np.round(n).astype(int)
n

This function samples matrices that have only 20 pairs but the co-occurences are _roughly_ equal to the join probabilities

In [ ]:
def sample_joint_n():
    
    high = [4, 3, 3]
    medium = [2,2 ,3]
    low = [1,1, 1]
    
    np.random.shuffle(high)
    np.random.shuffle(medium)
    
    n1 = [[high[0], medium[0], low[0]],
         [low[1], high[1], medium[1]],
         [medium[2], low[2], high[2]]]
    
    n2 = [[high[1], medium[1], low[0]],
         [low[1], high[2], medium[2]],
         [medium[0], low[2], high[0]]]
    
    n3 = [[high[2], medium[2], low[0]],
         [low[1], high[0], medium[0]],
         [medium[1], low[2], high[1]]]
    
    return np.array(n1), np.array(n2), np.array(n3)

In [ ]:
n1, n2, n3 = sample_joint_n()


n1, n2, n3, n1+n2+n3

# Problem 2

In [ ]:
def get_cost(pairs, last_pair=None):
    
    cost = (pairs[1:] == pairs[:-1]).any(1).sum()

    if last_pair is not None:
        
        cost += (pairs[0] == last_pair).any()
        
    return cost

In [ ]:
for i, j in zip(*map(np.ravel, np.meshgrid(range(3), range(3)))):
    print([(i,j)]*int(n[i, j]))

In [ ]:
def get_sequence(n, last_pair=None):
    i = 0
    
    pairs = []
    for i, j in zip(*map(np.ravel, np.meshgrid(range(3), range(3)))):
         pairs.append([(i, j)] *int(n[i, j]))

    pairs = np.concatenate(pairs)
    
    np.random.shuffle(pairs)
    
    with tqdm(range(10000)) as pbar:
        

        # Get cost of current pairs
        cost = get_cost(pairs)    
        for i in pbar:
            pbar.set_description(f'Current cost: {cost}')

            new_pairs = pairs.copy()

            # Flip 3 pairs so we don't get stuck in a local minimum
            rng = default_rng() 
            ix0, ix1, ix2 = rng.choice(20, size=3, replace=False)
            new_pairs[ix0], new_pairs[ix1], new_pairs[ix2] = pairs[ix2], pairs[ix0], pairs[ix1]

            new_cost = get_cost(new_pairs, last_pair)

            if new_cost<cost:
                pairs = new_pairs
                cost = new_cost

            if cost == 0.0:
                pbar.set_description(f'Current cost: {cost}')
                return pairs
            else:
                continue
    
        pairs = np.zeros((20,2))
        return pairs
            
        
        

In [ ]:
try:
    seq1 = get_sequence(n1)
    seq2 = get_sequence(n2, seq1[-1])
    seq3 = get_sequence(n3, seq2[-1])
    seq = np.concatenate((seq1, seq2, seq3))
except:
    pass

No repetitions...

In [ ]:
get_cost(seq)

This is what it looked like

In [ ]:
i = 0
n = np.array([[3, 3, 1],
        [1, 4, 2],
        [2, 1, 3]])

pairs = []
for i, j in zip(*map(np.ravel, np.meshgrid(range(3), range(3)))):
     pairs.append([(i, j)] *int(n[i, j]))

pairs = np.concatenate(pairs)
np.random.shuffle(pairs)
new_pairs = pairs.copy()
ix0, ix1, ix2 = np.random.randint(0, len(new_pairs), 3)
new_pairs[ix0], new_pairs[ix1], new_pairs[ix2] = pairs[ix2], pairs[ix0], pairs[ix1]

In [ ]:
new_pairs

In [ ]:
allTrialStructures = np.zeros((60,2,1))

In [ ]:
good_arrays = list()
for i in range(10000):
    n1, n2, n3 = sample_joint_n()
    seq1 = get_sequence(n1)
    seq2 = get_sequence(n2, seq1[-1])
    seq3 = get_sequence(n3, seq2[-1])
    if np.any(seq1) and np.any(seq2) and np.any(seq3):
        seq = np.concatenate((seq1, seq2, seq3))
    #if not any((seq == x).all() for x in good_arrays):
        good_arrays.append(list(seq))

In [ ]:
allTrialStructures = np.dstack((good_arrays))

In [ ]:
allTrialStructures.shape

In [ ]:
import scipy.io as io
#trial_dict = {"allTrialStructures": allTrialStructures, "label": "experiment"}
#scipy.io.savemat("allTrialStructures.mat",trial_dict)
io.savemat('newTenThousandStructures.mat', {"allTrialStructures":allTrialStructures})

In [ ]:
allTrialStructures[:,:,-1].shape